# 00 — Verify Connection

Confirms the three things every later notebook depends on:

1. **Spark / Databricks Connect** session is live (or you're in a Databricks workspace where `spark` exists)
2. **Unity Catalog** target schemas + the `raw_data` Volume exist (creates `{UC_SCHEMA}_bronze/_silver/_gold`)
3. **Synergy API** credentials work — exchanges client_id/secret for a token and pulls a few teams

Run this first. Configure `.env` from `.env.example` (laptop) or set a `synergy` secret scope (workspace).

In [ ]:
# Dual-mode setup: runs locally via Databricks Connect and inside a Databricks Git Folder.
import os, json
try:
    from dotenv import load_dotenv; load_dotenv()
except Exception:
    pass
try:
    spark  # already present inside a Databricks workspace notebook
except NameError:
    from databricks.connect import DatabricksSession
    spark = DatabricksSession.builder.serverless().getOrCreate()

UC_CATALOG    = os.getenv("UC_CATALOG", "main")
UC_SCHEMA     = os.getenv("UC_SCHEMA", "synergy_baseball_demo")
BRONZE_SCHEMA = f"{UC_SCHEMA}_bronze"
SILVER_SCHEMA = f"{UC_SCHEMA}_silver"
GOLD_SCHEMA   = f"{UC_SCHEMA}_gold"
VOLUME_NAME   = "raw_data"
VOLUME_PATH   = f"/Volumes/{UC_CATALOG}/{BRONZE_SCHEMA}/{VOLUME_NAME}"
B, S, G = f"{UC_CATALOG}.{BRONZE_SCHEMA}", f"{UC_CATALOG}.{SILVER_SCHEMA}", f"{UC_CATALOG}.{GOLD_SCHEMA}"
print("target:", B, "|", S, "|", G)

In [ ]:
from databricks.sdk import WorkspaceClient
w = WorkspaceClient()
spark.sql("SELECT current_user(), current_catalog(), current_timestamp()").show(truncate=False)
print("Spark:", spark.version)

In [ ]:
# medallion schemas + the raw landing Volume
for sch in [BRONZE_SCHEMA, SILVER_SCHEMA, GOLD_SCHEMA]:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {UC_CATALOG}.{sch}")
    print("  schema ready:", f"{UC_CATALOG}.{sch}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {UC_CATALOG}.{BRONZE_SCHEMA}.{VOLUME_NAME}")
print("  volume ready:", VOLUME_PATH)

## Synergy API probe

Resolves credentials and pulls a few teams. Two auth paths:
- **`SYNERGY_ACCESS_TOKEN`** in `.env`: a pre-issued bearer token (e.g. copied from the Synergy Swagger UI). Short-lived (~1h). Takes precedence when set.
- **`SYNERGY_CLIENT_ID` / `SYNERGY_CLIENT_SECRET`**: the OAuth2 client-credentials flow.

A 401 / `SynergyAPIError` here means the credentials aren't resolving. Fix that before moving on.

> **No credentials yet?** Skip this and run `tests/generate_fake_data.ipynb` to populate the Volume with synthetic payloads, then `02`/`03` run end-to-end offline.

In [ ]:
def get_secret(env_var, scope="synergy", key=None):
    """Resolve a credential from .env first, then a Databricks secret scope, so the same notebook
    runs from a laptop or inside a Databricks Git Folder. Synergy credentials live under scope
    `synergy`, keys `client_id` / `client_secret`. Never hardcode or commit them."""
    val = os.getenv(env_var)
    if val:
        return val
    from pyspark.dbutils import DBUtils  # type: ignore
    return DBUtils(spark).secrets.get(scope=scope, key=key or env_var.lower())

In [ ]:
from synergy_client import SynergyAPI

# Auth: prefer a pre-issued bearer token (SYNERGY_ACCESS_TOKEN), e.g. one copied from the Synergy
# Swagger UI (short-lived, ~1h). Otherwise use the OAuth client_id/secret flow.
access_token = os.getenv("SYNERGY_ACCESS_TOKEN")
if access_token:
    api = SynergyAPI(access_token=access_token)
    print("auth: using pre-issued SYNERGY_ACCESS_TOKEN")
else:
    api = SynergyAPI(
        client_id=get_secret("SYNERGY_CLIENT_ID", key="client_id"),
        client_secret=get_secret("SYNERGY_CLIENT_SECRET", key="client_secret"),
    )
    print("auth: using client_id/client_secret")

print("Token OK")
teams = api.get_teams()
print(f"Teams visible: {len(teams)}")
for t in teams[:5]:
    print(f"  - {t.get('id')}: {t.get('name')}  (MLBAM externalId={t.get('externalIdMlbam')})")

## Optional reset\nUncomment to wipe the demo schemas (keeps the Volume so you don't re-pull).

In [ ]:
# for sch in [BRONZE_SCHEMA, SILVER_SCHEMA, GOLD_SCHEMA]:
#     spark.sql(f"DROP SCHEMA IF EXISTS {UC_CATALOG}.{sch} CASCADE")
#     spark.sql(f"CREATE SCHEMA IF NOT EXISTS {UC_CATALOG}.{sch}")
# print("reset done")